In [1]:
!pip install pyspark

In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import pandas as pd

In [3]:
# Создание Spark сессии
spark = SparkSession.builder \
    .appName("IrisClassification") \
    .config("spark.master", "local") \
    .getOrCreate()

# Установка уровня логирования для уменьшения вывода
spark.sparkContext.setLogLevel("WARN")

In [4]:
# Загрузка встроенного датасета
from sklearn.datasets import load_iris

In [18]:
iris = load_iris()

In [19]:
# Создаем DataFrame из чистых Python данных
data = []
for i in range(len(iris.data)):
    data.append((
        float(iris.data[i][0]),  # sepal length
        float(iris.data[i][1]),  # sepal width
        float(iris.data[i][2]),  # petal length
        float(iris.data[i][3]),  # petal width
        str(iris.target_names[iris.target[i]])  # species
    ))

In [20]:
# Схема для Spark DataFrame
schema = StructType([
    StructField("sepal_length", DoubleType(), True),
    StructField("sepal_width", DoubleType(), True),
    StructField("petal_length", DoubleType(), True),
    StructField("petal_width", DoubleType(), True),
    StructField("species", StringType(), True)
])

In [21]:
# Создаем Spark DataFrame
csv_df = spark.createDataFrame(data, schema=schema)

print("Данные ирисов Фишера:")
csv_df.show(10)
print(f"Всего записей: {csv_df.count()}")

Данные ирисов Фишера:
+------------+-----------+------------+-----------+-------+
|sepal_length|sepal_width|petal_length|petal_width|species|
+------------+-----------+------------+-----------+-------+
|         5.1|        3.5|         1.4|        0.2| setosa|
|         4.9|        3.0|         1.4|        0.2| setosa|
|         4.7|        3.2|         1.3|        0.2| setosa|
|         4.6|        3.1|         1.5|        0.2| setosa|
|         5.0|        3.6|         1.4|        0.2| setosa|
|         5.4|        3.9|         1.7|        0.4| setosa|
|         4.6|        3.4|         1.4|        0.3| setosa|
|         5.0|        3.4|         1.5|        0.2| setosa|
|         4.4|        2.9|         1.4|        0.2| setosa|
|         4.9|        3.1|         1.5|        0.1| setosa|
+------------+-----------+------------+-----------+-------+
only showing top 10 rows

Всего записей: 150


In [22]:
# Проверка баланса классов
print("\nРаспределение по классам:")
csv_df.groupBy("species").count().show()


Распределение по классам:
+----------+-----+
|   species|count|
+----------+-----+
| virginica|   50|
|versicolor|   50|
|    setosa|   50|
+----------+-----+



In [23]:
# Подготовка фичей
assembler = VectorAssembler(
    inputCols=["sepal_length", "sepal_width", "petal_length", "petal_width"],
    outputCol="features"
)

# Преобразование строковых меток в числовые
label_indexer = StringIndexer(
    inputCol="species",
    outputCol="label"
)

In [24]:
# Модель. Логистическая регрессия
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0
)

# Создание пайплайна
pipeline = Pipeline(
    stages=[
        assembler,
        label_indexer,
        lr,
    ]
)

In [25]:
train_df, test_df = csv_df.randomSplit([0.8, 0.2], seed=42)

print(f"Тренировочная выборка: {train_df.count()} записей")
print(f"Тестовая выборка: {test_df.count()} записей")

# Обучение модели
print("\nОбучение модели...")
model = pipeline.fit(train_df)

# Предсказание на тестовой выборке
predictions = model.transform(test_df)

print("\nПример предсказаний:")
predictions.select("species", "label", "prediction", "probability").show(10)

# Оценка модели
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction"
)

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
precision = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
recall = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print("\n=== РЕЗУЛЬТАТЫ ОЦЕНКИ МОДЕЛИ ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

Тренировочная выборка: 126 записей
Тестовая выборка: 24 записей

Обучение модели...

Пример предсказаний:
+----------+-----+----------+--------------------+
|   species|label|prediction|         probability|
+----------+-----+----------+--------------------+
|    setosa|  2.0|       2.0|[0.02796319247481...|
|    setosa|  2.0|       2.0|[0.02374646586019...|
|    setosa|  2.0|       2.0|[0.00511898805052...|
|    setosa|  2.0|       2.0|[0.05296213073656...|
|    setosa|  2.0|       2.0|[0.05217009456322...|
|versicolor|  0.0|       0.0|[0.89017804864463...|
|    setosa|  2.0|       2.0|[0.02026274909661...|
|    setosa|  2.0|       2.0|[0.02343791787425...|
|    setosa|  2.0|       2.0|[0.01988961832168...|
|versicolor|  0.0|       0.0|[0.71875260761738...|
+----------+-----+----------+--------------------+
only showing top 10 rows


=== РЕЗУЛЬТАТЫ ОЦЕНКИ МОДЕЛИ ===
Accuracy: 1.0000
F1-score: 1.0000
Precision: 1.0000
Recall: 1.0000


In [26]:
# Сохранение модели
model_path = "/content/iris_logistic_regression_model"
model.write().overwrite().save(model_path)
print(f"\nМодель сохранена в: {model_path}")


Модель сохранена в: /content/iris_logistic_regression_model


In [27]:
from google.colab import files
import shutil

model_zip_path = "/content/iris_model.zip"
shutil.make_archive(model_zip_path.replace('.zip', ''), 'zip', model_path)

files.download(model_zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
# Демонстрация загрузки и использования сохраненной модели
from pyspark.ml import PipelineModel

print("\n=== ТЕСТИРОВАНИЕ СОХРАНЕННОЙ МОДЕЛИ ===")
loaded_model = PipelineModel.load(model_path)

# Создание тестовых данных
test_data = [
    (5.1, 3.5, 1.4, 0.2, "setosa"),    # Iris-setosa
    (6.7, 3.0, 5.2, 2.3, "virginica"), # Iris-virginica
    (5.9, 3.0, 4.2, 1.5, "versicolor"),# Iris-versicolor
]

test_df_demo = spark.createDataFrame(test_data, schema=schema)

# Предсказание с загруженной моделью
demo_predictions = loaded_model.transform(test_df_demo)

print("Предсказания для тестовых данных:")
demo_predictions.select("sepal_length", "sepal_width",
                       "petal_length", "petal_width", "species",
                       "prediction").show()

# Создание mapping для обратного преобразования предсказаний
label_mapping = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}

print("\nИнтерпретация предсказаний:")
for row in demo_predictions.collect():
    features = [row['sepal_length'], row['sepal_width'],
                row['petal_length'], row['petal_width']]
    pred = int(row['prediction'])
    species = label_mapping.get(pred, 'unknown')
    print(f"Признаки: {features} -> Предсказание: {species} (class {pred})")


=== ТЕСТИРОВАНИЕ СОХРАНЕННОЙ МОДЕЛИ ===
Предсказания для тестовых данных:
+------------+-----------+------------+-----------+----------+----------+
|sepal_length|sepal_width|petal_length|petal_width|   species|prediction|
+------------+-----------+------------+-----------+----------+----------+
|         5.1|        3.5|         1.4|        0.2|    setosa|       2.0|
|         6.7|        3.0|         5.2|        2.3| virginica|       1.0|
|         5.9|        3.0|         4.2|        1.5|versicolor|       0.0|
+------------+-----------+------------+-----------+----------+----------+


Интерпретация предсказаний:
Признаки: [5.1, 3.5, 1.4, 0.2] -> Предсказание: virginica (class 2)
Признаки: [6.7, 3.0, 5.2, 2.3] -> Предсказание: versicolor (class 1)
Признаки: [5.9, 3.0, 4.2, 1.5] -> Предсказание: setosa (class 0)


In [29]:
spark.stop()